In [7]:
import os
import pandas as pd
import numpy as np
import torch
#torchvision, torchaudio installed
from tqdm import tqdm

In [8]:
print(torch.__version__)
print(torch.backends.mps.is_available())
print(torch.backends.mps.is_built()) #checks if your current PyTorch installation was compiled with support for the MPS backend

device = torch.device("mps" if torch.mps.is_available() else "cpu")
print(f"Using device: {device}")
torch.mps.empty_cache()

2.10.0
True
True
Using device: mps


In [9]:
import warnings
import nltk
SEED = 0
torch.manual_seed(SEED)
torch.mps.manual_seed(SEED)
torch.use_deterministic_algorithms(True)
warnings.filterwarnings('ignore')
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/jessie_guo/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [10]:
from datasets import load_dataset
go_emotions = load_dataset("go_emotions")
print(go_emotions)
print(go_emotions['train'].features)

classes = go_emotions['train'].features['labels'].feature.names
print(f'\nClasses: {classes}')
len(classes)

train_data = pd.DataFrame(go_emotions['train'])
print(train_data)

counts = train_data['labels'].value_counts()
print(counts)
print(train_data['labels'])

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 43410
    })
    validation: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 5426
    })
    test: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 5427
    })
})
{'text': Value('string'), 'labels': List(ClassLabel(names=['admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval', 'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief', 'joy', 'love', 'nervousness', 'optimism', 'pride', 'realization', 'relief', 'remorse', 'sadness', 'surprise', 'neutral'])), 'id': Value('string')}

Classes: ['admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval', 'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief', 'joy', 'love', 'nervousness', 'optimism', 'pride', '

In [11]:
#EKMAN MAPPING
import json

with open('ekman_mapping.json') as f:
    ekman_mapping = json.load(f)

ekman_mapping['neutral'] = ["neutral"]

ekman_classes = list(ekman_mapping.keys())

ekman2id = {e: i for i, e in enumerate(ekman_classes)}

ekman_num = {
    0: "anger", 1: "anger", 2: "anger",  # anger, annoyance, disapproval
    3: "disgust",                         # disgust
    4: "fear", 5: "fear",                 # fear, nervousness
    6: "joy", 7: "joy", 8: "joy", 9: "joy", 10: "joy", 11: "joy", 12: "joy", 13: "joy", 14: "joy", 15: "joy", 16: "joy", 17: "joy",  # joy etc.
    18: "sadness", 19: "sadness", 20: "sadness", 21: "sadness", 22: "sadness",  # sadness etc.
    23: "surprise", 24: "surprise", 25: "surprise", 26: "surprise", # surprise etc.
    27: "neutral"  # neutral
}

print(ekman_mapping)
print(ekman_classes)

def map_to_ekman(example):
    ekman_labels = set()

    for label in example["labels"]:
        ekman_category = ekman_num[label]
        ekman_labels.add(ekman_category)
        
    multi_hot = [0] * len(ekman_classes)
    for e in ekman_labels:
        multi_hot[ekman2id[e]] = 1

    example["labels"] = multi_hot
    return example
         
reduced_emotions = go_emotions.map(map_to_ekman)

train_data = pd.DataFrame(reduced_emotions['train'])
print(train_data)

counts = train_data['labels'].value_counts()
print(counts)

{'anger': ['anger', 'annoyance', 'disapproval'], 'disgust': ['disgust'], 'fear': ['fear', 'nervousness'], 'joy': ['joy', 'amusement', 'approval', 'excitement', 'gratitude', 'love', 'optimism', 'relief', 'pride', 'admiration', 'desire', 'caring'], 'sadness': ['sadness', 'disappointment', 'embarrassment', 'grief', 'remorse'], 'surprise': ['surprise', 'realization', 'confusion', 'curiosity'], 'neutral': ['neutral']}
['anger', 'disgust', 'fear', 'joy', 'sadness', 'surprise', 'neutral']
                                                    text  \
0      My favourite food is anything I didn't have to...   
1      Now if he does off himself, everyone will thin...   
2                         WHY THE FUCK IS BAYLESS ISOING   
3                            To make her feel threatened   
4                                 Dirty Southern Wankers   
...                                                  ...   
43405  Added you mate well I’ve just got the bow and ...   
43406  Always thought that was fu

In [12]:
print(reduced_emotions['train'][0:8])

{'text': ["My favourite food is anything I didn't have to cook myself.", 'Now if he does off himself, everyone will think hes having a laugh screwing with people instead of actually dead', 'WHY THE FUCK IS BAYLESS ISOING', 'To make her feel threatened', 'Dirty Southern Wankers', "OmG pEyToN iSn'T gOoD eNoUgH tO hElP uS iN tHe PlAyOfFs! Dumbass Broncos fans circa December 2015.", 'Yes I heard abt the f bombs! That has to be why. Thanks for your reply:) until then hubby and I will anxiously wait 😝', 'We need more boards and to create a bit more space for [NAME]. Then we’ll be good.'], 'labels': [[0, 0, 0, 0, 0, 0, 1], [0, 0, 0, 0, 0, 0, 1], [1, 0, 0, 0, 0, 0, 0], [0, 0, 0, 1, 0, 0, 0], [0, 1, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 1, 0], [0, 0, 0, 1, 0, 0, 0], [0, 0, 0, 1, 1, 0, 0]], 'id': ['eebbqej', 'ed00q6i', 'eezlygj', 'ed7ypvh', 'ed0bdzj', 'edvnz26', 'ee3b6wu', 'ef4qmod']}


In [18]:
#data processing EKMAN CLASSES VERS.
from transformers import AutoTokenizer, DataCollatorWithPadding
PROBLEM_TYPE == "multi_label_classification"
model_checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def tokenize_labelling(examples):
    tokenized = tokenizer(
        examples['text'],
        truncation = True,
        padding = "max_length",
        max_length = 128)
    return tokenized

if PROBLEM_TYPE == "single_label_classification":
    def onehot_to_class(example):
        example["labels"] = int(np.argmax(example["labels"]))
    return example
    
    single_label_emotions = reduced_emotions.map(onehot_to_class)
    print(single_label_emotions["train"][0]["labels"])

def problem_type(reduced_dataset):
    tokenized_go_emotions = reduced_dataset.map(tokenize_labelling, 
                                        batched = True, 
                                        remove_columns = ["text"])
    return tokenized_go_emotions

if PROBLEM_TYPE == "single_label_classification":
    tokenized_go_emotions = problem_type(single_label_emotions)
elif PROBLEM_TYPE == "multi_label_classification":
    tokenized_go_emotions = problem_type(reduced_emotions)

print(pd.DataFrame(tokenized_go_emotions['train']))

tokenized_go_emotions.set_format(
    type = "torch",
    columns = ["input_ids", "attention_mask", "labels"])

print(pd.DataFrame(tokenized_go_emotions['train']))
print(type(tokenized_go_emotions["train"][0]["labels"]))

class CustomDataCollator: #Only for multi-label classification
    def __call__(self, batch):
        input_ids = torch.stack([x["input_ids"] for x in batch])
        attention_mask = torch.stack([x["attention_mask"] for x in batch])
        labels = torch.stack([x["labels"] for x in batch]).to(torch.float32)  # important!
        return {"input_ids": input_ids,
                "attention_mask": attention_mask,
                "labels": labels}

data_collator = CustomDataCollator()
#dynamically pads each batch to longest sequence in batch rather than padding every sequence - saves compute, memory
#mlm=False to avoid randomly masking tokens

6
                      labels       id  \
0      [0, 0, 0, 0, 0, 0, 1]  eebbqej   
1      [0, 0, 0, 0, 0, 0, 1]  ed00q6i   
2      [1, 0, 0, 0, 0, 0, 0]  eezlygj   
3      [0, 0, 0, 1, 0, 0, 0]  ed7ypvh   
4      [0, 1, 0, 0, 0, 0, 0]  ed0bdzj   
...                      ...      ...   
43405  [0, 0, 0, 0, 1, 0, 0]  edsb738   
43406  [0, 0, 0, 1, 0, 0, 0]  ee7fdou   
43407  [0, 1, 0, 0, 0, 0, 0]  efgbhks   
43408  [0, 0, 0, 1, 0, 0, 0]  ed1naf8   
43409  [0, 0, 0, 1, 0, 0, 0]  eecwmbq   

                                               input_ids  \
0      [101, 2026, 8837, 2833, 2003, 2505, 1045, 2134...   
1      [101, 2085, 2065, 2002, 2515, 2125, 2370, 1010...   
2      [101, 2339, 1996, 6616, 2003, 3016, 3238, 1116...   
3      [101, 2000, 2191, 2014, 2514, 5561, 102, 0, 0,...   
4      [101, 6530, 2670, 14071, 11451, 102, 0, 0, 0, ...   
...                                                  ...   
43405  [101, 2794, 2017, 6775, 2092, 1045, 1521, 2310...   
43406  [101, 2467, 2245, 

In [10]:
#Weights adjusted
y_train = np.array(tokenized_go_emotions["train"]["labels"])
pos_counts = y_train.sum(axis=0)
neg_counts = len(y_train) - pos_counts
pos_weight = neg_counts/pos_counts
pos_weight = torch.tensor(pos_weight, dtype = torch.float)
print(pos_weight)

tensor([ 4.4804, 16.5749,  9.9594,  2.2568,  7.7839, 13.3837,  2.0530])


In [ ]:
#Weights adjusted
import torch.nn as nn
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback

class WeightedTrainer(Trainer): #creates new class inheriting from Trainer
    def __init__(self, *args, pos_weight=None, **kwargs): #*args passes any number of positional arguments, **kwargs pass any number of keyword (named) arguments. 
        super().__init__(*args, **kwargs) #calls original Trainer
        self.pos_weight = pos_weight

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs): #rewrites the fn in Trainer
        labels = inputs.pop("labels").float() #removes target answer from inputs dict, need to reserve for validation
        outputs = model(**inputs)
        logits = outputs.logits #raw scores model produces before turned into probabilities

        loss_fct = nn.BCEWithLogitsLoss( #stand. loss for multi-labelling
            pos_weight = self.pos_weight.to(logits.device) #if rare label missed, multiply penalty by pos_weight, .to ensures weights moved to same loc as model
        )
        loss = loss_fct(logits, labels)

        return (loss, outputs) if return_outputs else loss #in case Trainer needs predictions for eval metrics

In [169]:
from sklearn.metrics import f1_score, precision_score, recall_score
#For multi-label classification
def find_best_threshold(logits, labels):
    probs = 1/(1 + np.exp(-logits))
    best_t_micro = 0.5
    best_t_macro = 0.5
    best_t_weighted = 0.5
    best_f1_micro = 0
    best_f1_macro = 0
    best_f1_weighted = 0

    for t in np.arange(0.1, 0.9, 0.05):
        preds = (probs > t).astype(int)
        f1_micro = f1_score(labels, preds, average = "micro", zero_division = 0)
        f1_macro = f1_score(labels, preds, average = "macro", zero_division = 0)
        f1_weighted = f1_score(labels, preds, average = "weighted", zero_division = 0)
        if f1_micro > best_f1_micro:
            best_f1_micro = f1_micro
            best_t_micro = t
        if f1_macro > best_f1_macro:
            best_f1_macro = f1_macro
            best_t_macro = t
        if f1_weighted > best_f1_weighted:
            best_f1_weighted = f1_weighted
            best_t_weighted = t

    return best_t_macro, best_t_micro, best_t_weighted, best_f1_macro, best_f1_micro, best_f1_weighted

In [ ]:
import torch.nn as nn
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, precision_recall_fscore_support, hamming_loss

PROBLEM_TYPE = "multi_label_classification"

def compute_metrics(eval_pred):
    logits, labels = eval_pred

    if PROBLEM_TYPE == "multi_label_classification":

        best_t_macro, best_t_micro, best_t_weighted, best_f1_macro, best_f1_micro, best_f1_weighted = find_best_threshold(logits, labels)
    
        probs = 1 / (1 + np.exp(-logits)) #remove for multi-class classification
    
        predictions = (probs > best_t_weighted).astype(int)  # Get the predicted class labels

        hamming_loss_score = hamming_loss(labels, predictions)

    elif PROBLEM_TYPE == "single_label_classification":

        predictions = logits.argmax(axis=-1)  # Get the predicted class labels

    precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(labels, predictions, average='weighted', zero_division=0)
    f1_micro = f1_score(labels, predictions, average = "micro", zero_division = 0)
    f1_macro = f1_score(labels, predictions, average = "macro", zero_division = 0)
    precision_micro = precision_score(labels, predictions, average="micro", zero_division=0)
    recall_micro = recall_score(labels, predictions, average="micro", zero_division=0)
    precision_macro = precision_score(labels, predictions, average="macro", zero_division=0)
    recall_macro = recall_score(labels, predictions, average="macro", zero_division=0)

    metrics_dict = {
        'f1_weighted': f1_weighted,
        'precision_weighted': precision_weighted,
        'recall_weighted': recall_weighted,
        'f1_micro': f1_micro,
        'precision_micro': precision_micro,
        'recall_micro': recall_micro,
        'f1_macro': f1_macro,
        'precision_macro': precision_macro,
        'recall_macro': recall_macro,
    }

    if PROBLEM_TYPE == "multi_label_classification":
        metrics_dict.update({
            'hamming_loss_score': hamming_loss_score,
            'best_threshold': best_t_weighted
        })
    
    return metrics_dict

def training_model(TRAIN_OUTPUT, SAVE_OUPUT):
    model_1 = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, 
                                                                 num_labels = len(ekman_classes), #len(classes),  
                                                                 problem_type = PROBLEM_TYPE) #"single_label_classification" for multi-class classification #load the weights in saved dtype. Without, doubles memory usage if weights originally torch.bfloat16
    model_1.to(device)
    
    training_args = TrainingArguments(
        output_dir = TRAIN_OUTPUT,
        num_train_epochs = 10,
        per_device_train_batch_size = 32,
        gradient_accumulation_steps = 4, #accumulate gradients across several batches and update once as opposed to updating weights every batch, simulates training with large batch size w/o storing, saves memory but slower training
        #gradient_checkpointing=True, #recomputes intermediate activations during forward pass as opposed to storing - extra computation but saves memory
        fp16 = False, #bf16 for fast mixed precision training, fp16 false for mps
        learning_rate = 2e-5, #initial lr
        logging_steps = 100, #controls how frequently to update + return loss
        eval_strategy = "epoch", #when to evaluate a model during training
        save_strategy = "epoch", #when to save
        weight_decay = 0.01, #L2 regularisation, helps prevent model from overfitting by discouraging large weights - uses AdamW, independent of weight updates --> better training stability
        metric_for_best_model = 'eval_f1_weighted',
        greater_is_better = True,
        load_best_model_at_end = True,
        save_total_limit = 2
    )
    
    trainer_kwargs = {
        "model": model_1,
        "args": training_args,
        "train_dataset": tokenized_go_emotions["train"],
        "eval_dataset": tokenized_go_emotions["validation"],
        "processing_class": tokenizer,
        "compute_metrics": compute_metrics,
        "callbacks": [EarlyStoppingCallback(early_stopping_patience=3)],
        #pos_weight: pos_weight
    }
    
    if PROBLEM_TYPE == "multi_label_classification":
        trainer_kwargs["data_collator"] = data_collator

    trainer = Trainer(**trainer_kwargs)    
    trainer.train()
    
    trainer.save_model(SAVE_OUPUT)
    tokenizer.save_pretrained(SAVE_OUPUT)

return trainer

trainer_multilabel = training_model("expdistilbert_finetuned-goemotions_multilabel", "expdistilbert_finetuned-goemotions-final_multilabel")


In [ ]:
PROBLEM_TYPE = "single_label_classification"
trainer_multiclass = training_model("expdistilbert_finetuned-goemotions_multiclass", "expdistilbert_finetuned-goemotions-final_multiclass")

In [15]:
#Training and Validation Loss, Weighted_F1 Curves
import matplotlib.pyplot as plt
import pandas as pd

def loss_curves_and_weighted_f1(trainer):
    trainer_multilabel = Multilabel
    trainer_multiclass = Multiclass
    history = trainer.state.log_history
    df = pd.DataFrame(history)
    
    train_df = df[df['loss'].notna()]
    eval_df = df[df['eval_loss'].notna()]
    weighted_f1_df = df[df['eval_f1_weighted'].notna()]
    
    plt.figure(figsize=(10, 6))
    
    plt.plot(train_df['epoch'], train_df['loss'], label='Training Loss')
    plt.plot(eval_df['epoch'], eval_df['eval_loss'], label='Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.savefig(f"Training and Validation Loss {trainer[-10:]}")
    plt.clf()
    
    plt.plot(weighted_f1_df['epoch'], weighted_f1_df['eval_f1_weighted'])
    plt.xlabel('Epochs')
    plt.ylabel('Weighted F1 Score')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.savefig(f"Weighted F1 Score {trainer[-10:]}")

loss_curves_and_weighted_f1(trainer_multilabel)
loss_curves_and_weighted_f1(trainer_multiclass)


NameError: name 'trainer' is not defined

In [17]:
import torch.nn as nn
from transformers import AutoModelForSequenceClassification, AutoTokenizer, TrainingArguments, Trainer, EarlyStoppingCallback

SAVE_OUTPUT = "distilbert_finetuned-goemotions-final_multilabel"

def save_to_device(SAVE_OUTPUT):
    model_new = AutoModelForSequenceClassification.from_pretrained(SAVE_OUTPUT)
    tokenizer_new = AutoTokenizer.from_pretrained(SAVE_OUTPUT)
    return model_new, tokenizer_new

model_new, tokenizer_new = save_to_device(SAVE_OUTPUT)

model_new.to(device)
model_new.eval()

Loading weights: 100%|█████████████████████████████████████████████████| 104/104 [00:00<00:00, 8286.30it/s]


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [28]:
from torch.utils.data import Dataset
class CustomDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len):
        self.tokenizer = tokenizer
        self.dataframe = dataframe
        self.texts = dataframe['text']
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):
        text = str(self.texts.iloc[index])
        text = " ".join(text.split())
        inputs = self.tokenizer(
            text,
            None,
            add_special_tokens=True,  # Add special tokens
            max_length=self.max_len,
            padding='max_length',  # Pad to max_length
            return_token_type_ids=True,
            return_tensors='pt',  # Return PyTorch tensors
            truncation=True  # Truncate sequences longer than max_length
        )
        
        input_ids = inputs['input_ids'].squeeze(0)  # Remove the added batch dimension
        attention_mask = inputs['attention_mask'].squeeze(0)  # Remove the added batch dimension
        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
        }

In [29]:
def test():
    model_new.eval()
    all_outputs = []
    with torch.no_grad():
        for data in test_loader:
            input_ids = data['input_ids'].to(device) #, dtype=torch.long)
            attention_mask = data['attention_mask'].to(device) #, dtype=torch.long)
            outputs = model_new(input_ids = input_ids, attention_mask = attention_mask)
            logits = outputs.logits

            if PROBLEM_TYPE == "multi_label_classification":
                probs = torch.sigmoid(logits)
                all_outputs.extend(probs.cpu().numpy().tolist())
            elif PROBLEM_TYPE == "single_label_classification":
                probs = torch.softmax(logits, dim = -1)
                predictions = torch.argmax(probs, dim = -1)
                all_outputs.extend(predictions.cpu().numpy().tolist())  
    
    return all_outputs

In [25]:
lengths = [len(tokenizer_new.encode(text)) for text in go_emotions['train']['text']]
print(max(lengths)) #max token count across all texts
print("95th percentile:", np.percentile(lengths, 95)) #95% of samples contain 34 tokens or fewer 

MAX_LEN = 128

316
95th percentile: 34.0


In [21]:
#preliminary test evaluations
texts_path = 'Georgian Texts (Old Repo)/Vazha Pshavela/'
sources = ['en/', 'ggl/', 'gem/', 'gpt/']
poems_path = texts_path + 'Poems/revised/'
poem_names = []

output_base = "results/poems/sentiment/"
os.makedirs(output_base, exist_ok=True)

ge_poems_directory = os.fsencode(poems_path + 'ge/')
for file in sorted(os.listdir(ge_poems_directory)):
    file = file.decode()
    if file.endswith(".md"):
        poem_names.append(file)

In [30]:
from torch.utils.data import DataLoader

for source in sources:
    output_dir = os.path.join(output_base, source)
    os.makedirs(output_dir, exist_ok = True)
    
    for poem in poem_names:
        input_file = os.path.join(poems_path, source, poem)
        
        poem_text = [line.strip() for line in open(input_file, "r")]
        poem_df = pd.DataFrame(poem_text, columns=['text'])
        test_dataset = CustomDataset(poem_df, tokenizer_new, MAX_LEN)
        test_params = {'batch_size': 1, 'shuffle': False, 'num_workers': 0}
        test_loader = DataLoader(test_dataset, **test_params)
        
        if PROBLEM_TYPE == "multi_label_classification":
            test_probs = np.array(test())
        
            test_outputs = (test_probs >= 0.3).astype(int)    
        
            for j, label in enumerate(ekman_classes):
                poem_df[label] = test_outputs[:, j]
        
        elif PROBLEM_TYPE == "single_label_classification":
            test_predictions = np.array(test())

            poem_df["predicted_label_id"] = test_predictions
    
            poem_df["predicted_label"] = [ekman_classes[i] for i in test_predictions]

        output_file = os.path.join(output_dir, poem)
        poem_df.to_csv(output_file)

In [230]:
#multi-label
t01 = pd.read_csv("results/poems/sentiment/en/06_that_in_truth_is_not_manliness.md")
t01_text = t01["text"]
print(t01_text)

if PROBLEM_TYPE == "multi_label_classification":

    t01["manual_label"] = [
        ["neutral", "disgust"],
        ["neutral", "disgust"],
        ["neutral", "disgust"],
        ["neutral"],
        ["anger"],
        ["neutral"],
        ["disgust", "neutral"],
        ["neutral", "disgust"],
        ["disgust"],
        ["neutral"]
    ]
    
    for label in ekman_classes:
        t01[f"manual_{label}"] = 0
    
    for i, labels_for_row in enumerate(manual_labels):
        for label in labels_for_row:
            if label in ekman_classes:
                t01.at[i, f"manual_{label}"] = 1
            else:
                print(f"Invalid label in row {i}: {label}")
    
    manual_cols = [f"manual_{label}" for label in ekman_classes]
    pred_cols = ekman_classes
    print(t01)
    
    y_true = t01[manual_cols].astype(int).values
    y_pred = t01[pred_cols].astype(int).values

elif PROBLEM_TYPE == "single_label_classification":
    t01["manual_label"] = [
    "neutral",
    "neutral",
    "neutral",
    "neutral",
    "anger",
    "neutral",
    "disgust",
    "disgust",
    "disgust",
    "neutral",
]

    label_to_id = {label: i  for i, label in enumerate (ekman_classes)}
    
    t01["manual_label_id"] = manual_df["manual_label"].map(label_to_id)
    print(t01)
    
    y_true = t01["manual_label_id"].values
    y_pred = t01["predicted_label_id"].values

precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted', zero_division=0)
f1_micro = f1_score(y_true, y_pred, average = "micro", zero_division = 0)
f1_macro = f1_score(y_true, y_pred, average = "macro", zero_division = 0)
precision_micro = precision_score(y_true, y_pred, average="micro", zero_division=0)
recall_micro = recall_score(y_true, y_pred, average="micro", zero_division=0)
precision_macro = precision_score(y_true, y_pred, average="macro", zero_division=0)
recall_macro = recall_score(y_true, y_pred, average="macro", zero_division=0)

print(
    'f1_weighted:', f1_weighted,
    '\nprecision_weighted:', precision_weighted,
    '\nrecall_weighted:', recall_weighted,
    '\nf1_micro:', f1_micro,
    '\nprecision_micro:', precision_micro,
    '\nrecall_micro:', recall_micro,
    '\nf1_macro:', f1_macro,
    '\nprecision_macro:', precision_macro,
    '\nrecall_macro:', recall_macro,
)

0                That in truth is not manliness When someone eggs you on and makes you brash.
1                 A man would I call you only if Of your own inclination you shew your worth.
2               Nor to call it manliness would I wish When you throttle the one you vanquish.
3                 A man would I call you only when One throttled by another you were to tend.
4             Tell me, who has styled it manly virtue When it is oppressors that you nurture?
5                              Rather is it true manliness When you suffer for the oppressed.
6                         And no manliness begins When with full stomach you sing your hymns,
7    Or when you behave as superior to others, And because of this despise us, your brothers,
8                  When from your minstrelsy you give none peace But ever vaunt and overheat.
9          A man then surely would I call you Were you to sing in spite of thirst and hunger.
Name: text, dtype: object
   Unnamed: 0  \
0           0   
